In [6]:
# ==========================================
# ANÁLISE: PERGUNTAS DO PROJETO
# ==========================================
import sys
import pandas as pd
from IPython.display import display

# ==========================================
# 1. ORQUESTRAÇÃO DA FÁBRICA (BACKGROUND)
# ==========================================
import main
print("🏭 Acionando a Fábrica de Dados (ETL) em Background...")
conn = main.orquestrar_pipeline()

if conn is None:
    print("❌ Falha ao orquestrar a fábrica. Abortando análises.")
else:
    print("\n" + "="*50)
    print("✅ Banco de Dados conectado! Iniciando extração de insights de negócio...")
    print("="*50 + "\n")

    # ==========================================
    # 2. PERGUNTA 1: VULNERABILIDADE POR COMPANHIA
    # ==========================================
    print("🔍 PERGUNTA 1: Taxa de cancelamento por Companhia em dias de Chuva/Tempestade")
    query_analise_1 = """
        SELECT
            cia.[Companhia Nome],
            COUNT(f.FLAG_CANCELADO) AS TOTAL_VOOS,
            SUM(f.FLAG_CANCELADO) AS TOTAL_CANCELADOS,
            ROUND(CAST(SUM(f.FLAG_CANCELADO) AS FLOAT) / COUNT(f.FLAG_CANCELADO) * 100, 2) AS TAXA_CANCELAMENTO_PCT
        FROM fato_voos f
        INNER JOIN dim_cia_aerea cia ON f.ID_CIA_AEREA = cia.ID_CIA_AEREA
        INNER JOIN dim_clima c ON f.ID_CLIMA = c.ID_CLIMA
        WHERE c.CONDICAO IN ('Chuva', 'Tempestade')
        GROUP BY cia.[Companhia Nome]
        ORDER BY TAXA_CANCELAMENTO_PCT DESC
    """
    df_analise_1 = pd.read_sql_query(query_analise_1, conn)
    display(df_analise_1)

    print("\n" + "="*50 + "\n")

    # ==========================================
    # 3. PERGUNTA 3: VULNERABILIDADE POR TURNO
    # ==========================================
    print("🔍 PERGUNTA 3: Qual turno sofre mais cancelamentos sob Chuva/Tempestade?")
    query_analise_3 = """
        SELECT
            CASE
                WHEN t.HORA >= 6 AND t.HORA < 12 THEN '1. Manhã (06h-11h)'
                WHEN t.HORA >= 12 AND t.HORA < 18 THEN '2. Tarde (12h-17h)'
                WHEN t.HORA >= 18 AND t.HORA <= 23 THEN '3. Noite (18h-23h)'
                ELSE '4. Madrugada (00h-05h)'
            END AS TURNO,
            c.CONDICAO,
            COUNT(f.FLAG_CANCELADO) AS TOTAL_VOOS,
            SUM(f.FLAG_CANCELADO) AS TOTAL_CANCELADOS
        FROM fato_voos f
        INNER JOIN dim_tempo t ON f.ID_TEMPO = t.ID_TEMPO
        INNER JOIN dim_clima c ON f.ID_CLIMA = c.ID_CLIMA
        WHERE c.CONDICAO IN ('Chuva', 'Tempestade')
        GROUP BY TURNO, c.CONDICAO
        ORDER BY TURNO, c.CONDICAO
    """
    df_analise_3 = pd.read_sql_query(query_analise_3, conn)
    display(df_analise_3)

🏭 Acionando a Fábrica de Dados (ETL) em Background...
🚀 INICIANDO OPERAÇÃO: PIPELINE PONTE AÉREA
📥 Iniciando extração dos dados da ANAC...
✅ Extração VRA concluída: 86637 registros encontrados.
🌦️ Fazendo o disparo para a API de Rio de Janeiro...
✅ Clima de Rio de Janeiro extraído com sucesso.
🌦️ Fazendo o disparo para a API de São Paulo...
✅ Clima de São Paulo extraído com sucesso.
⚙️ Iniciando Transformação e Modelagem Star Schema...
✅ Transformação concluída. Tabela Fato forjada (Limpa de redundâncias).
🗄️ Iniciando ingestão no banco de dados SQLite (em memória)...
✅ Carga Star Schema concluída. Banco de dados pronto e blindado.
📊 Validando a Integridade no Banco de Dados:
              CONDICAO_CLIMATICA  TOTAL_VOOS  CANCELADOS
0                          Chuva        4084         232
1                      Céu Limpo         418          26
2                     Tempestade         582          40
3  ⚠️ ALERTA: SEM DADOS DE CLIMA         148           0
🎯 MISSÃO CUMPRIDA! O Pipeline 

,Companhia Nome,TOTAL_VOOS,TOTAL_CANCELADOS,TAXA_CANCELAMENTO_PCT
0,Gol Linhas Aéreas,1744,116,6.65
1,Azul Linhas Aéreas,1256,68,5.41
2,LATAM Airlines,1666,88,5.28




🔍 PERGUNTA 3: Qual turno sofre mais cancelamentos sob Chuva/Tempestade?


,TURNO,CONDICAO,TOTAL_VOOS,TOTAL_CANCELADOS
0,1. Manhã (06h-11h),Chuva,1577,79
1,1. Manhã (06h-11h),Tempestade,212,4
2,2. Tarde (12h-17h),Chuva,1422,73
3,2. Tarde (12h-17h),Tempestade,202,12
4,3. Noite (18h-23h),Chuva,1037,79
5,3. Noite (18h-23h),Tempestade,159,22
6,4. Madrugada (00h-05h),Chuva,48,1
7,4. Madrugada (00h-05h),Tempestade,9,2
